<a href="https://colab.research.google.com/github/mbenedicto99/Canopus/blob/main/ImagePDF2TXT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Primeiro, instalamos a biblioteca PyPDF2
# Se você já tiver ela instalada, pode pular esta célula.
%pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.2 MB/s eta 0:00:00


In [3]:
from google.colab import files
import os

# Esta função permite que você faça upload de arquivos diretamente para o ambiente Colab
uploaded = files.upload()

# A variável 'uploaded' é um dicionário, onde as chaves são os nomes dos arquivos
# Se você carregou apenas um arquivo, pegaremos o primeiro nome do arquivo.
if uploaded:
    for filename in uploaded.keys():
        print(f'Arquivo "{filename}" carregado com sucesso!')
        # Aqui você pode definir pdf_file_path para o arquivo carregado
        # Para usar este arquivo, você precisará copiar o nome do arquivo abaixo
        # e colá-lo na célula de extração de texto, substituindo 'nome_do_seu_arquivo.pdf'
        uploaded_pdf_filename = filename
        print(f'O nome do arquivo carregado é: {uploaded_pdf_filename}')
        # Uma vez que o arquivo é carregado, você pode processá-lo.
        # Por exemplo, para usar com o código PyPDF2, você faria algo como:
        # pdf_file_path = uploaded_pdf_filename
        # print(f'Por favor, atualize a variável pdf_file_path na próxima célula com: {pdf_file_path}')
else:
    print('Nenhum arquivo foi carregado.')

Saving Micro_texto.pdf to Micro_texto.pdf
Arquivo "Micro_texto.pdf" carregado com sucesso!
O nome do arquivo carregado é: Micro_texto.pdf


In [7]:
# 1. Instalar Poppler (ferramenta para converter PDF em imagem)
# Isso é uma dependência do sistema, então usamos apt-get
!apt-get install poppler-utils
# Instalar o pacote de idioma português para Tesseract OCR
!apt-get install tesseract-ocr-por

# 2. Instalar as bibliotecas Python para OCR
# pdf2image para converter páginas do PDF em imagens
# pytesseract para realizar o OCR
# Pillow (PIL) já deve estar instalada na maioria dos ambientes Colab, mas garantimos.
%pip install pdf2image pytesseract Pillow

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tesseract-ocr-por
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 856 kB of archives.
After this operation, 1,998 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-por all 1:4.00~git30-7274cfa-1.1 [856 kB]
Fetched 856 kB in 1s (666 kB/s)
Selecting previously unselected package tesseract-ocr-por.
(Reading database ... 118224 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-por_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-por (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-

Agora que as ferramentas estão instaladas, vamos usar o `pdf2image` para converter o PDF em uma lista de imagens e, em seguida, aplicar o `Pytesseract` para extrair o texto de cada imagem.

In [8]:
from PIL import Image
import pytesseract
from pdf2image import convert_from_path

# Caminho para o arquivo PDF que você carregou anteriormente
# Usaremos a variável `uploaded_pdf_filename` do upload anterior
# Se você não carregou um arquivo ou reiniciou o ambiente,
# você precisará carregar novamente o arquivo ou definir `pdf_file_path` manualmente.
pdf_file_path = uploaded_pdf_filename # Garante que estamos usando o arquivo carregado

text_ocr = ''
try:
    # Converte cada página do PDF em uma imagem PIL (Pillow Image)
    # `poppler_path` pode ser necessário em alguns ambientes, mas no Colab geralmente funciona sem.
    images = convert_from_path(pdf_file_path)

    print(f'Convertendo {len(images)} páginas para imagens e aplicando OCR...')

    for i, image in enumerate(images):
        # Realiza OCR na imagem da página e adiciona ao texto total
        page_text = pytesseract.image_to_string(image, lang='por') # 'lang' define o idioma do OCR
        text_ocr += page_text + '\n---\nPágina {i+1}\n---\n'

    print(f'Texto extraído por OCR (primeiras 500 caracteres):\n{text_ocr[:500]}')
    print(f'Total de caracteres extraídos por OCR: {len(text_ocr)}')

except FileNotFoundError:
    print(f"Erro: O arquivo '{pdf_file_path}' não foi encontrado. Verifique o caminho e o nome do arquivo.")
except Exception as e:
    print(f"Ocorreu um erro ao processar o PDF com OCR: {e}")

# Se quiser ver todo o texto extraído por OCR, descomente a linha abaixo
# print(text_ocr)

Convertendo 9 páginas para imagens e aplicando OCR...
Texto extraído por OCR (primeiras 500 caracteres):
98 5. THE IDEAL NEOCLASSICAL MARKET AND GENERAL EQUILIBRIUM

OUTLINE

5.1 Introduction: The Neoclassical Paradigm
and Research Program 98
5.2 Consumer Theory 100
5.2.1 Preferences 100
5.2.2 Utility, Maximization, and Demand 102
An Illustration 103
5.2.3 Marshallian Demand, Hicksian
Demand, and the Slutsky
Equation 105
5.3 Production Theory 106
5.3.1 The Production Function 106
5.3.2 Cost Minimization and Cost
Functions 108
5.3.3 Profit Maximization 109
5.4 Partial Equilibrium 111
5.5 General Equ
Total de caracteres extraídos por OCR: 27823


In [12]:
# Instalar a biblioteca python-docx
%pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.8 MB/s eta 0:00:00


In [14]:
from docx import Document
import re # Importa o módulo de expressões regulares

# Certifique-se de que text_ocr contém o texto extraído do PDF
# Se a célula de OCR não foi executada ou não gerou texto, text_ocr pode estar vazio.
if text_ocr:
    # Limpa o texto para remover caracteres não compatíveis com XML
    # Isso remove a maioria dos caracteres de controle e bytes nulos
    cleaned_text_ocr = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', text_ocr)

    document = Document()
    document.add_paragraph(cleaned_text_ocr)

    output_filename = 'saida_ocr.docx'
    document.save(output_filename)
    print(f'O texto do OCR foi salvo em "{output_filename}" com sucesso!')

    # Você pode baixar o arquivo se estiver rodando no Google Colab
    from google.colab import files
    files.download(output_filename)
else:
    print('Não há texto para salvar. Por favor, execute a célula de OCR primeiro.')

O texto do OCR foi salvo em "saida_ocr.docx" com sucesso!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>